In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, fbeta_score,
    confusion_matrix, classification_report
)

# =========================
# DEAL-JÄGER (komplett)
# =========================
# Erklärung: Ziel = "Deal-Jäger" (Top-20% Rabattanteil). Danach:
# - Leakage raus (Deals/Anteil nicht als Features)
# - Modell trainieren (LogReg + class_weight)
# - Threshold optimieren (F0.5 -> Precision wichtiger)
# - Deal-Score + Flag für alle Kunden erstellen

# 1) Daten laden
df = pd.read_csv("Marktkampagne.csv")

# 2) Mini-Clean
df["Datum_Kunde"] = pd.to_datetime(df["Datum_Kunde"], dayfirst=True, errors="raise")
df["Einkommen_fehlte"] = df["Einkommen"].isna().astype(int)
df["Einkommen"] = df["Einkommen"].fillna(df["Einkommen"].median())

# 3) Hilfsgrößen für Deal-Target
df["Kaeufe_Gesamt"] = df["Anzahl_Webkäufe"] + df["Anzahl_Katalogkäufe"] + df["Anzahl_Ladeneinkäufe"]
df["Rabattanteil"] = (
    df["Anzahl_Rabattkäufe"] / df["Kaeufe_Gesamt"].replace(0, np.nan)
).clip(upper=1.0).fillna(0.0)

# 4) Deal-Target bauen (Top-20% Rabattanteil)
deal_thr = df["Rabattanteil"].quantile(0.80)
df["Ziel_DealJaeger"] = (df["Rabattanteil"] >= deal_thr).astype(int)

print("Deal-Schwelle (Rabattanteil):", float(deal_thr))
print("Deal-Rate (Anteil 1):", df["Ziel_DealJaeger"].mean())

# 5) Features bauen (Leakage vermeiden)
# Erklärung: Alles, was direkt Deals/Anteil enthält, raus – sonst wäre es "zu leicht".
drop_cols = [
    "ID", "Datum_Kunde", "Ziel_DealJaeger",
    "Anzahl_Rabattkäufe", "Rabattanteil", "Kaeufe_Gesamt"
]
X_deal = df.drop(columns=[c for c in drop_cols if c in df.columns])
y_deal = df["Ziel_DealJaeger"]

cat_cols = X_deal.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in X_deal.columns if c not in cat_cols]

# 6) Train/Test Split
Xtr, Xte, ytr, yte = train_test_split(
    X_deal, y_deal, test_size=0.2, random_state=42, stratify=y_deal
)

# 7) Modell (Baseline)
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

model_deal = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))
])

model_deal.fit(Xtr, ytr)
proba_deal = model_deal.predict_proba(Xte)[:, 1]

print("\nROC-AUC:", roc_auc_score(yte, proba_deal))
print("PR-AUC:", average_precision_score(yte, proba_deal))

# 8) Threshold-Optimierung (F0.5 -> Precision wichtiger)
thresholds = np.linspace(0.05, 0.95, 19)

best = {"t": None, "precision": None, "recall": None, "f1": None, "f0_5": -1, "positives_pred": None}
for t in thresholds:
    pred = (proba_deal >= t).astype(int)
    p = precision_score(yte, pred, zero_division=0)
    r = recall_score(yte, pred, zero_division=0)
    f1 = f1_score(yte, pred, zero_division=0)
    f05 = fbeta_score(yte, pred, beta=0.5, zero_division=0)

    if f05 > best["f0_5"]:
        best = {"t": float(t), "precision": p, "recall": r, "f1": f1, "f0_5": f05, "positives_pred": int(pred.sum())}

pred_best = (proba_deal >= best["t"]).astype(int)

print("\nBeste Schwelle (F0.5):", best["t"])
print({"precision": best["precision"], "recall": best["recall"], "f1": best["f1"], "f0_5": best["f0_5"], "positives_pred": best["positives_pred"]})
print("Confusion Matrix:\n", confusion_matrix(yte, pred_best))
print("\nReport:\n", classification_report(yte, pred_best))

# 9) Deal-Score + Flag für alle Kunden (für spätere Nutzung)
proba_all = model_deal.predict_proba(X_deal)[:, 1]
df["Deal_Score"] = proba_all
df["Deal_Flag"] = (df["Deal_Score"] >= best["t"]).astype(int)

# 10) Top-Liste anzeigen
deal_top = df.loc[df["Deal_Flag"] == 1, ["ID", "Deal_Score"]].sort_values("Deal_Score", ascending=False)

print("\nTop Deal-Jäger (erste 20):")
display(deal_top.head(20))


Deal-Schwelle (Rabattanteil): 0.4
Deal-Rate (Anteil 1): 0.22321428571428573

ROC-AUC: 0.9249137931034482
PR-AUC: 0.7321203404801626

Beste Schwelle (F0.5): 0.85
{'precision': 0.8448275862068966, 'recall': 0.49, 'f1': 0.620253164556962, 'f0_5': 0.7379518072289156, 'positives_pred': 58}
Confusion Matrix:
 [[339   9]
 [ 51  49]]

Report:
               precision    recall  f1-score   support

           0       0.87      0.97      0.92       348
           1       0.84      0.49      0.62       100

    accuracy                           0.87       448
   macro avg       0.86      0.73      0.77       448
weighted avg       0.86      0.87      0.85       448


Top Deal-Jäger (erste 20):


,ID,Deal_Score
148,5885,0.997297
383,3310,0.997297
1846,9931,0.996553
216,7264,0.996435
137,9579,0.994465
311,2826,0.994213
613,6222,0.994213
607,8477,0.993511
2129,10104,0.993208
1558,6642,0.993008


In [ ]:
# =========================
# Erklärung (Deal-Jäger Ergebnis – einfach)
# =========================
# 1) Ziel/Label:
#    - Wir nennen einen Kunden "Deal-Jäger", wenn sein Rabattanteil zu den obersten ~20% gehört.
#    - Deal-Schwelle (Rabattanteil) = 0.40 bedeutet: ab ca. 40% Rabattanteil zählt er als Deal-Jäger.
#    - Deal-Rate = 0.223 heißt: ca. 22% der Kunden im Datensatz sind Deal-Jäger (nach dieser Definition).
#
# 2) Modellgüte (wie gut trennt das Modell generell?):
#    - ROC-AUC = 0.925 -> sehr gute Trennschärfe (das Modell kann Deal-Jäger gut von Nicht-Deal-Jägern unterscheiden).
#    - PR-AUC = 0.732 -> ebenfalls gut, speziell für den "positiven" Fall (Deal-Jäger) aussagekräftig.
#
# 3) Optimierte Entscheidungsschwelle (wie streng markieren wir Deal-Jäger?):
#    - Beste Schwelle (F0.5) = 0.85
#      -> Wir markieren nur Kunden als Deal-Jäger, wenn das Modell mindestens 85% Wahrscheinlichkeit sieht.
#      -> F0.5 bedeutet: Precision ist wichtiger als Recall (Rabatte kosten Geld, also lieber weniger Fehlalarme).
#
# 4) Was bedeutet Precision/Recall hier?
#    - Precision (0.84): Von allen Kunden, die wir als Deal-Jäger markieren, sind ca. 84% wirklich Deal-Jäger.
#      -> sehr wenig "unnötige Rabatte" / Streuverlust.
#    - Recall (0.49): Wir finden ca. 49% aller echten Deal-Jäger.
#      -> wir lassen einige Deal-Jäger bewusst weg, um präziser zu sein.
#
# 5) Confusion Matrix (konkret in Zahlen):
#    - [[339,  9],
#       [ 51, 49]]
#    - 49 = echte Deal-Jäger korrekt erkannt (TP)
#    -  9 = fälschlich als Deal-Jäger markiert (FP)  -> sehr wenig
#    - 51 = Deal-Jäger verpasst (FN)                  -> passiert wegen strenger Schwelle
#    - 339 = korrekt als Nicht-Deal-Jäger erkannt (TN)
#
# 6) positives_pred = 58:
#    - Das Modell markiert 58 Kunden im Test als Deal-Jäger.
#    - Das ist eine "kleine, hochwertige Zielgruppe": wenig Leute, aber hohe Trefferquote.
#
# Kurzfazit:
# - Das Modell ist insgesamt stark (AUC hoch).
# - Die Optimierung zielt auf "wenig Fehlalarme" ab (Precision-first).
# - Ergebnis: gute Trefferquote (84%), aber nicht alle Deal-Jäger werden erwischt (49% Recall).
